# Flow rate and HFNO failure: propensity-matched analysis

The exposure is fixed at a landmark so that it cannot depend on how long HFNO lasted. The maximum
flow over the whole episode rises with both duration and deterioration during therapy, so it is
post-baseline, and balancing pre-initiation covariates cannot remove that.

1. Exposure: mean charted O2 flow over the first 2 h of HFNO (`flow_rate_2h`, from notebook 1).
   Only patients still on HFNO at 2 h are analysed, so the exposure is complete before any outcome.
   The maximum over the episode is retained for Table 1 only and is not used as the exposure.
2. Split at the cohort median of the 2-hour flow rate (high flow = at or above the median).
3. Propensity score for receiving high flow: logistic regression on all pre-initiation variables
   (the same variables as the pre-HFNO-only model).
4. 1:1 nearest-neighbour matching on the logit of the propensity score, without replacement,
   caliper 0.2 SD (Austin 2011). Balance checked with standardised mean differences (SMD < 0.1).
5. Primary: failure proportions, risk difference (bootstrap CI over matched pairs) and odds ratio.
   Secondary: Kaplan-Meier with log-rank test and a Cox model from HFNO initiation.

Matching balances pre-initiation variables only. Deterioration after HFNO start is not accounted
for, so the estimate is an association and not a causal effect of flow.

In [ ]:
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

# No database connection: the exposure comes from notebook 1's saved flow_rate_2h column.
os.makedirs('results', exist_ok=True)
CALIPER_SD = 0.2
SEED = 42
N_BOOT = 2000

## 1. Cohort and outcome timing

In [ ]:
df = pd.read_csv('./filtered_data/mimiciv_tabledata_imputed.csv', index_col=0)
ids = pd.read_csv('./processed_data/mimiciv_hfno_data.csv', usecols=['stay_id', 'final_starttime', 'final_endtime'])

# The analysis table has no stay_id; HFNO start/end times identify each stay uniquely.
key = ['final_starttime', 'final_endtime']
assert not ids.duplicated(key).any()
df = df.merge(ids, on=key, how='left', validate='one_to_one')
assert df['stay_id'].notna().all()
print(f"Cohort: {len(df)} stays, {df.hfno_failure.sum()} failures")

start = pd.to_datetime(df['final_starttime'])
hours = lambda t: (pd.to_datetime(t) - start).dt.total_seconds() / 3600
df['time_fail_h'] = np.where(
    df['hfno_failure'] == 1,
    np.where(df['intubated'] == 1, hours(df['iv_time']), hours(df['deathtime'])),
    hours(df['final_endtime']),
)
df['time_fail_h'].describe()

## 2. Exposure: flow rate over the first 2 hours (landmark)

In [ ]:
# `flow_rate_2h` is the mean charted O2 flow over the first 2 h of HFNO, extracted in notebook 1.
assert 'flow_rate_2h' in df.columns, 'flow_rate_2h missing - rerun notebook 1'

# Landmark: keep only patients still on HFNO at 2 h, so the exposure is measured before any
# outcome can have occurred and does not depend on episode duration.
df['hfno_hours'] = ((pd.to_datetime(df['final_endtime']) - pd.to_datetime(df['final_starttime']))
                    .dt.total_seconds() / 3600)
n_before = len(df)
df = df[df.hfno_hours >= 2].copy()
print(f"Landmark at 2 h: {len(df)} of {n_before} stays retained, {int(df.hfno_failure.sum())} failures")
print(f"Stays with a 2 h flow value: {df.flow_rate_2h.notna().sum()} / {len(df)}")
df['flow_rate_2h'].describe()

In [ ]:
FLOW_THRESHOLD = df['flow_rate_2h'].median()
print(f"Median flow rate over the first 2 h: {FLOW_THRESHOLD:.0f} L/min")
fl = df[df.flow_rate_2h.notna()].copy()
fl['high_flow'] = (fl['flow_rate_2h'] >= FLOW_THRESHOLD).astype(int)
fl.groupby('high_flow').agg(n=('stay_id', 'size'), failures=('hfno_failure', 'sum'),
                            failure_rate=('hfno_failure', 'mean'),
                            median_flow_2h=('flow_rate_2h', 'median'))

## 3. Propensity score from pre-initiation variables

In [ ]:
non_covariates = {'stay_id', 'final_starttime', 'final_endtime', 'deathtime', 'iv_time', 'hfno_failure',
                  'intubated', 'died', 'inhosp_mortality', 'time_fail_h', 'max_flow_rate', 'high_flow',
                  'hfno_hours', 'ps', 'logit_ps', 'flow_rate_2h', 'fio2_2h', 'rox_2h'}
covariates = [c for c in fl.columns if c not in non_covariates]
# drop one reference level per one-hot group to avoid perfect collinearity
covariates = [c for c in covariates if c not in ('race_cat_WHITE', 'gcs_binned_Normal (15)')]
assert not any(c in covariates for c in ('hfno_hours', 'time_fail_h', 'flow_rate_2h', 'max_flow_rate')), \
    'post-baseline variable in the propensity score'
print(f"{len(covariates)} covariates:", covariates)

X = StandardScaler().fit_transform(fl[covariates])
ps_model = LogisticRegression(max_iter=5000, C=1.0).fit(X, fl['high_flow'])
fl['ps'] = ps_model.predict_proba(X)[:, 1]
fl['logit_ps'] = np.log(fl['ps'] / (1 - fl['ps']))
fl.groupby('high_flow')['ps'].describe()

## 4. 1:1 nearest-neighbour matching (caliper 0.2 SD of logit PS)

In [ ]:
caliper = CALIPER_SD * fl['logit_ps'].std()
treated = fl[fl.high_flow == 1].sample(frac=1, random_state=SEED)   # random matching order
control = fl[fl.high_flow == 0].copy()

available = set(control.index)
pairs = []
nn = NearestNeighbors(n_neighbors=min(50, len(control))).fit(control[['logit_ps']].to_numpy())
for idx, row in treated.iterrows():
    dist, nbr = nn.kneighbors([[row['logit_ps']]])
    for d, j in zip(dist[0], nbr[0]):
        cidx = control.index[j]
        if d > caliper:
            break
        if cidx in available:
            available.remove(cidx)
            pairs.append((idx, cidx))
            break

matched = pd.concat([fl.loc[[t for t, _ in pairs]], fl.loc[[c for _, c in pairs]]])
print(f"Caliper: {caliper:.3f} (logit PS) | treated: {len(treated)} | matched pairs: {len(pairs)}")

In [ ]:
def smd(a, b):
    sd = np.sqrt((a.var() + b.var()) / 2)
    return 0.0 if sd == 0 else (a.mean() - b.mean()) / sd

balance = pd.DataFrame({
    'SMD before': [smd(fl.loc[fl.high_flow == 1, c], fl.loc[fl.high_flow == 0, c]) for c in covariates],
    'SMD after':  [smd(matched.loc[matched.high_flow == 1, c], matched.loc[matched.high_flow == 0, c]) for c in covariates],
}, index=covariates).round(3)
print(f"Covariates with |SMD| > 0.1: before {int((balance['SMD before'].abs() > 0.1).sum())}, "
      f"after {int((balance['SMD after'].abs() > 0.1).sum())}")
balance.to_csv('results/Flow_rate_PSM_balance.csv')
balance.reindex(balance['SMD before'].abs().sort_values(ascending=False).index).head(15)

## 5. HFNO failure: high vs low flow at 2 hours

The binary contrast and the time-to-event contrast answer different questions and will not agree.
HFNO success is treated as censoring at discontinuation, so patients weaned early contribute
little time at risk; the crude proportions ignore that, while the hazard ratio accounts for it.
Both are reported, with median follow-up, so the difference is visible rather than puzzling.

In [ ]:
def flow_summary(d, label, paired=False, n_boot=N_BOOT, seed=SEED):
    hi, lo = d[d.high_flow == 1], d[d.high_flow == 0]
    rd = hi.hfno_failure.mean() - lo.hfno_failure.mean()
    rng = np.random.default_rng(seed)
    if paired:      # resample matched pairs, keeping each pair intact
        h, l = hi.hfno_failure.to_numpy(), lo.hfno_failure.to_numpy()
        boots = [(h[i].mean() - l[i].mean())
                 for i in (rng.integers(0, len(h), len(h)) for _ in range(n_boot))]
    else:
        boots = [(rng.choice(hi.hfno_failure.to_numpy(), len(hi)).mean()
                  - rng.choice(lo.hfno_failure.to_numpy(), len(lo)).mean())
                 for _ in range(n_boot)]
    rd_lo, rd_hi = np.percentile(boots, [2.5, 97.5])
    lg = sm.Logit(d.hfno_failure, sm.add_constant(d.high_flow.astype(float))).fit(disp=0)
    or_, (or_lo, or_hi) = np.exp(lg.params['high_flow']), np.exp(lg.conf_int().loc['high_flow'])
    lr = logrank_test(hi.time_fail_h, lo.time_fail_h,
                      event_observed_A=hi.hfno_failure, event_observed_B=lo.hfno_failure)
    s = CoxPHFitter().fit(d[['time_fail_h', 'hfno_failure', 'high_flow']],
                          'time_fail_h', 'hfno_failure').summary.loc['high_flow']
    return {'Cohort': label, 'n': len(d),
            'failures high/low': f"{int(hi.hfno_failure.sum())}/{int(lo.hfno_failure.sum())}",
            'failure rate high': round(hi.hfno_failure.mean(), 3),
            'failure rate low': round(lo.hfno_failure.mean(), 3),
            'risk difference (95% CI)': f"{rd * 100:.1f} ({rd_lo * 100:.1f} to {rd_hi * 100:.1f})",
            'risk ratio': round(hi.hfno_failure.mean() / lo.hfno_failure.mean(), 2),
            'OR (95% CI)': f"{or_:.2f} ({or_lo:.2f}-{or_hi:.2f})", 'OR p': lg.pvalues['high_flow'],
            'HR (95% CI)': f"{s['exp(coef)']:.2f} ({s['exp(coef) lower 95%']:.2f}-{s['exp(coef) upper 95%']:.2f})",
            'Cox p': s['p'], 'log-rank p': lr.p_value,
            'median follow-up high/low (h)': f"{hi.time_fail_h.median():.0f}/{lo.time_fail_h.median():.0f}"}


# matched pairs in the same order (treated i <-> control i) for the paired bootstrap
matched_paired = pd.concat([fl.loc[[t for t, _ in pairs]], fl.loc[[c for _, c in pairs]]])
flow_results = pd.DataFrame([flow_summary(fl, 'Unmatched'),
                             flow_summary(matched_paired, 'Propensity-matched', paired=True)])
flow_results['threshold (L/min)'] = FLOW_THRESHOLD
flow_results['exposure'] = 'mean flow rate over the first 2 h of HFNO'
flow_results.to_csv('results/Flow_rate_PSM_results.csv', index=False)
flow_results

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
kmf = KaplanMeierFitter()
for grp, color, label in [(0, '#1E88E5', f'Flow at 2 h < {FLOW_THRESHOLD:.0f} L/min'),
                          (1, '#E91E63', f'Flow at 2 h \u2265 {FLOW_THRESHOLD:.0f} L/min')]:
    g = matched[matched.high_flow == grp]
    kmf.fit(g.time_fail_h, g.hfno_failure, label=label).plot_survival_function(ax=ax, color=color)
r = flow_results.iloc[1]
ax.text(0.70, 0.30, f"HR = {r['HR (95% CI)']}\nlog-rank p = {r['log-rank p']:.3g}",
        transform=ax.transAxes, ha='center', va='center', fontsize=11,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.85, edgecolor='gray'))
ax.set_xlabel('Time elapsed since HFNO initiation (hours)', fontsize=13)
ax.set_ylabel('Probability of remaining free of HFNO failure', fontsize=13)
ax.set_title(f'Propensity-matched cohort ({len(pairs)} pairs)', fontsize=12)
ax.grid(True, linestyle='--', alpha=0.35)
plt.tight_layout()
plt.savefig('results/Flow_rate_KM_PSM.png', dpi=300)
plt.savefig('results/Flow_rate_KM_PSM.pdf')

# matched cohort (survival data only) for the manuscript figure in notebook 6
matched[['high_flow', 'time_fail_h', 'hfno_failure']].to_csv('results/Flow_rate_matched_cohort.csv', index=False)
plt.show()